# Lab06: Writing Data in Neo4j

Santiago Elí Jiménez Aguilar
Luis Eduardo Gonzalez Gloria

## Goal:
Crear un pipeline de datos para analizar el dataset de **Recommendation Videogames** usando GraphFrames y persistirlo en **Neo4j**.

## Instrucciones
- Descargar el dataset desde: https://networkrepository.com/rec-amz-Video-Games.php
- Ingesta con PySpark
- Análisis de grafo (PageRank, Label Propagation, Triangle Count, Degree Distribution)
- Escritura en Neo4j (corregido para evitar bloqueos)
- Captura de pantalla del grafo en Neo4j

In [1]:
from spark_utils import SparkUtils

neo4j_connector = "org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3,io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5"
su = SparkUtils("Lab06: Writing Data in Neo4j", "spark://spark-master:7077", spark_packages=neo4j_connector)
su.spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.neo4j#neo4j-connector-apache-spark_2.13 added as a dependency
io.graphframes#graphframes-spark3_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-79514fb2-9c71-4b8f-bd75-2563c6aa9f93;1.0
	confs: [default]
	found org.neo4j#neo4j-connector-apache-spark_2.13;5.3.10_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.13_common;5.3.10_for_spark_3 in central
	found org.neo4j#caniuse-core;1.3.0 in central
	found org.neo4j#caniuse-api;1.3.0 in central
	found org.jetbrains.kotlin#kotlin-stdlib;2.1.20 in central
	found org.jetbrains#annotations;13.0 in central
	found org.neo4j#caniuse-neo4j-detection;1.3.0 in central
	found org.neo4j.driver#neo4j-java-driver-slim;4.4.21 in central
	found org.reactivestreams#reactiv

## 1. Data Ingestion

In [2]:
from graphframes import GraphFrame
from pyspark.sql import functions as F

# Schema del dataset
mvideo_games_schema = SparkUtils.generate_schema([
    ("userId",      "string"),
    ("videoGameId", "string"),
    ("rating",      "float"),
    ("timestamp",   "long")
])

# Lectura del dataset
video_games_df = (su.spark.read
                .option("header", "false")
                .schema(mvideo_games_schema)
                .csv("/opt/spark/work-dir/data/rec-amz-Video-Games"))

# Filtrar posibles líneas corruptas
video_games_df = video_games_df.filter(
    ~F.col("userId").startswith("<") &
    ~F.col("videoGameId").startswith("<")
)

# Crear vértices
user_vertices = video_games_df.select(
    F.col("userId").alias("id"),
    F.lit("user").alias("type")
).distinct()

game_vertices = video_games_df.select(
    F.col("videoGameId").alias("id"),
    F.lit("game").alias("type")
).distinct()

vertices = user_vertices.union(game_vertices)

# Crear aristas
edges = video_games_df.select(
    F.col("userId").alias("src"),
    F.col("videoGameId").alias("dst"),
    F.col("rating").alias("rating"),
    F.col("timestamp").alias("timestamp")
)

# Crear GraphFrame
g = GraphFrame(vertices, edges)

print(f"Vertices: {g.vertices.count()}")
print(f"Edges: {g.edges.count()}")
g.vertices.show(5)
g.edges.show(5)

/usr/local/lib/python3.10/dist-packages/pyspark/sql/classic/dataframe.py:146: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
                                                                                

Vertices: 876977


Edges: 1324753


+--------------+----+
|            id|type|
+--------------+----+
| AE7GUHCDQQ4UI|user|
|A26B0P6K95SIKW|user|
|A182S3ANC0W7DL|user|
|A1T98OCCYW6OBI|user|
|A1TBUSGCBTXWFC|user|
+--------------+----+
only showing top 5 rows
+--------------+----------+------+----------+
|           src|       dst|rating| timestamp|
+--------------+----------+------+----------+
| AB9S9279OZ3QO|0078764343|   5.0|1373155200|
|A24SSUT5CSW8BH|0078764343|   5.0|1377302400|
| AK3V0HEBJMQ7J|0078764343|   4.0|1372896000|
|A10BECPH7W8HM7|043933702X|   5.0|1404950400|
|A2PRV9OULX1TWP|043933702X|   5.0|1386115200|
+--------------+----------+------+----------+
only showing top 5 rows


## 2. Graph Analysis

### PageRank

In [3]:
results = g.pageRank(resetProbability=0.15, maxIter=5)

print("=== TOP 10 USUARIOS POR PAGERANK ===")
results.vertices \
    .filter(F.col("type") == "user") \
    .select("id", "type", "pagerank") \
    .orderBy("pagerank", ascending=False) \
    .show(10, truncate=False)

print("=== TOP 10 VIDEOJUEGOS POR PAGERANK ===")
results.vertices \
    .filter(F.col("type") == "game") \
    .select("id", "type", "pagerank") \
    .orderBy("pagerank", ascending=False) \
    .show(10, truncate=False)

/usr/local/lib/python3.10/dist-packages/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


=== TOP 10 USUARIOS POR PAGERANK ===


+---------------------+----+------------------+
|id                   |type|pagerank          |
+---------------------+----+------------------+
|A0009878M2RGMMHGJH39 |user|0.5551439694748398|
|A0002090WKEMAO8KOWKM |user|0.5551439694748398|
|A00063061AK7XBIZLCOXJ|user|0.5551439694748398|
|A00101847G3FJTWYGNQA |user|0.5551439694748398|
|A00101961G0VS92WDGJ11|user|0.5551439694748398|
|A00278362652PLQL0XWQ5|user|0.5551439694748398|
|A00096001PYDTZQQ42NU6|user|0.5551439694748398|
|A00593941PIA0QVHHIONI|user|0.5551439694748398|
|A001147626R4BL248CZ5T|user|0.5551439694748398|
|A00933513755ZZ7REP527|user|0.5551439694748398|
+---------------------+----+------------------+
only showing top 10 rows
=== TOP 10 VIDEOJUEGOS POR PAGERANK ===


+----------+----+------------------+
|id        |type|pagerank          |
+----------+----+------------------+
|B00DJFIMW6|game|7285.526431395484 |
|B00BGA9WK2|game|2645.507195083908 |
|B00FAX6XQC|game|2547.3341328377564|
|B009KS4XRO|game|2501.0385889444806|
|B0055SWM08|game|1989.4563258041183|
|B00CSR2J9I|game|1964.373283273377 |
|B002VBWIP6|game|1857.1934441848766|
|B0015AARJI|game|1408.3694893308366|
|B000FKBCX4|game|1243.5554664378442|
|B00178630A|game|1230.274656615952 |
+----------+----+------------------+
only showing top 10 rows


### Label Propagation

In [4]:
lpa = g.labelPropagation(maxIter=3)
lpa.select("id", "type", "label").show(20, truncate=False)

[Stage 466:=============================================>           (4 + 1) / 5]

+---------------------+----+-----------+
|id                   |type|label      |
+---------------------+----+-----------+
|043933702X           |game|25769804420|
|0439671418           |game|8589972610 |
|0439900581           |game|17179871748|
|0545115507           |game|55201      |
|0700026657           |game|8590005570 |
|1886846758           |game|17179911574|
|7293000936           |game|8590098049 |
|7542614444           |game|25769958148|
|9078439122           |game|34359872010|
|9861064222           |game|17179908478|
|986118452X           |game|8590070871 |
|9861767304           |game|8589942402 |
|986325083X           |game|25769883582|
|9941113300           |game|94401      |
|A0009878M2RGMMHGJH39 |user|169495     |
|A00101961G0VS92WDGJ11|user|8590100259 |
|A001147626R4BL248CZ5T|user|25769977765|
|A0011756FPL8K71Q5TAQ |user|34359907769|
|A00230923E4Y7VHWZK0IC|user|8590107813 |
|A00338543M2OZPUWO9ZRU|user|172460     |
+---------------------+----+-----------+
only showing top

### Triangle Counting

In [5]:
triangle_count = g.triangleCount()
triangle_count.select("id", "type", "count").show(20)

[Stage 584:=============================================>           (4 + 1) / 5]

+--------------+----+-----+
|            id|type|count|
+--------------+----+-----+
| AE7GUHCDQQ4UI|user|    0|
|A1TBUSGCBTXWFC|user|    0|
|A129SW886TYQ6H|user|    0|
| AFWPLXT2OD6H1|user|    0|
|A26B0P6K95SIKW|user|    0|
|A366EUKI8WMGYB|user|    0|
|A10N7L0GMRODUO|user|    0|
|A261CI99KET69W|user|    0|
|A214Z566V6QEDF|user|    0|
|A182S3ANC0W7DL|user|    0|
|A2M64UKVOU9CWU|user|    0|
|A2890J1FHE76RY|user|    0|
| AJRHPTQ7TXPD6|user|    0|
|A1NC9PQCE3NOGA|user|    0|
|A27QRTHZLBA61M|user|    0|
| AA9LU15A9PX9E|user|    0|
|A1T98OCCYW6OBI|user|    0|
|A2NQXA21O64HDZ|user|    0|
| ADOCLYEFV2PKH|user|    0|
| A8SSL0QMV2VY1|user|    0|
+--------------+----+-----+
only showing top 20 rows


### Degree Distribution

In [6]:
print("=== In-Degree (juegos más populares) ===")
in_deg = g.inDegrees.join(vertices, "id")
in_deg.orderBy("inDegree", ascending=False).show(10)

print("=== Out-Degree (usuarios más activos) ===")
out_deg = g.outDegrees.join(vertices, "id")
out_deg.orderBy("outDegree", ascending=False).show(10)

=== In-Degree (juegos más populares) ===


+----------+--------+----+
|        id|inDegree|type|
+----------+--------+----+
|B00DJFIMW6|   16221|game|
|B00BGA9WK2|    7561|game|
|B00FAX6XQC|    5713|game|
|B009KS4XRO|    5489|game|
|B002VBWIP6|    5190|game|
|B0055SWM08|    4638|game|
|B00CSR2J9I|    4510|game|
|B0015AARJI|    4468|game|
|B00178630A|    3522|game|
|B000FKBCX4|    3290|game|
+----------+--------+----+
only showing top 10 rows
=== Out-Degree (usuarios más activos) ===


[Stage 607:======================================>                  (2 + 1) / 3]

+--------------+---------+----+
|            id|outDegree|type|
+--------------+---------+----+
|A3V6Z4RCDGRC44|      880|user|
|A3W4D8XOGLWUN5|      817|user|
| AJKWF4W7QD4NS|      797|user|
|A2QHS1ZCIQOL7E|      521|user|
|A2TCG2HV1VJP6V|      474|user|
|A29BQ6B90Y1R5F|      429|user|
| AFV2584U13XP3|      338|user|
|A20DZX38KRBIT8|      320|user|
| A74TA8X5YQ7NE|      267|user|
|A2582KMXLK2P06|      263|user|
+--------------+---------+----+
only showing top 10 rows


## 3. Writing Data in Neo4j

In [ ]:
neo4j_url    = "bolt://neo4j-iteso:7687"
neo4j_user   = "neo4j"
neo4j_passwd = "neo4j@1234"

# ====================== ESCRIBIR NODOS ======================

# Usuarios
g.vertices.filter(F.col("type") == "user").write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("labels", ":User") \
  .option("node.keys", "id") \
  .save()

# Videojuegos
g.vertices.filter(F.col("type") == "game").write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("labels", ":Game") \
  .option("node.keys", "id") \
  .save()

print("Nodos (:User y :Game) escritos correctamente en Neo4j")

# ====================== ESCRIBIR RELACIONES ======================

print("Escribiendo relaciones RATED... (esto puede tardar varios minutos)")

g.edges \
  .repartition(1) \
  .write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("relationship", "RATED") \
  .option("relationship.save.strategy", "keys") \
  .option("relationship.source.labels", ":User") \
  .option("relationship.source.save.mode", "match") \
  .option("relationship.source.node.keys", "src:id") \
  .option("relationship.target.labels", ":Game") \
  .option("relationship.target.save.mode", "match") \
  .option("relationship.target.node.keys", "dst:id") \
  .option("batch.size", "10000") \
  .option("transaction.retries", "20") \
  .save()

print("Vertices y edges escritos exitosamente en Neo4j!")

[Stage 610:>                                                        (0 + 1) / 3]

## 4. Querying the Graph

**Instrucciones:**
1. Abre Neo4j Browser
2. Ejecuta la siguiente consulta:

```cypher
MATCH (u:User)-[r:RATED]->(g:Game)
RETURN u, r, g
LIMIT 200
```

3. Toma una captura de pantalla del grafo visualizado y pégala aquí.

In [ ]:
su.spark.stop()